# Pairwise Fold Distribution Shift Analysis for Lo-Hi Hi Datasets

This notebook analyzes pairwise fold distribution shift for the Hi datasets used in the valid OOD-holdout vs random-shuffle protocol comparison: DRD2-Hi, HIV-Hi and Sol-Hi.

KDR-Hi is treated only as a diagnostic special case. Its outer training folds are restricted to 500 molecules and its reconstructed test subsets are not disjoint, so it is not included in the main shift-vs-protocol-benefit analysis.

## Methodology: Shift Detection as Binary Classification

We treat distribution-shift detection as a binary classification problem. For each dataset, we reconstruct the three original subsets as `F1 = test_3`, `F2 = test_2`, and `F3 = test_1`, then build the three pairwise discrimination tasks: `F1_vs_F2`, `F1_vs_F3`, and `F2_vs_F3`. In each task, molecules from the first subset receive label 0 and molecules from the second subset receive label 1. The activity label is not used.

Two complementary discrimination regimes are evaluated:

- **In-sample setting (high-capacity):** Estimates maximum separability using weakly regularized discriminators. This is only a diagnostic: if a model cannot separate two folds even in-sample, the two subsets are strongly overlapping in the chosen representation.
- **Out-of-sample setting:** Estimates generalizable shift using the same model families and hyperparameter search spaces used in the main OOD-vs-random protocol comparison. This second setting is the main shift estimate.

## Models and Representations

The discriminators are **Decision Tree (DT)**, **Logistic Regression (LR)**, and **Linear SVM**. Molecular representations include **ECFP4**, **MACCS**, and **RDKit descriptors**, although the main plots use only ECFP4 and MACCS to keep model/fingerprint comparisons aligned across DT, LR, and SVM. RDKit descriptors are retained in the CSV outputs for additional analysis. Continuous RDKit descriptors are scaled inside a pipeline; binary fingerprints are not scaled.

## The Shift Score

Throughout this study, we report the **shift score**:

$$s = 2 \cdot \text{balanced accuracy} - 1 \in [0,1]$$

Here, $s=0$ means that the two folds are not distinguishable by the discriminator, while $s=1$ means perfect distinguishability. This score is a monotonic rescaling of the proxy $\mathcal{A}$-distance from the domain-adaptation literature:

$$\hat{d}_{\mathcal{A}} = 2(1 - 2\epsilon)$$

Where $\epsilon$ is the discriminator error. Since balanced accuracy is $1-\epsilon$, we have:

$$s = \frac{\hat{d}_{\mathcal{A}}}{2}$$

> **Note:** We use $s$ in figures for readability and report $\hat{d}_{\mathcal{A}}$ in the CSV outputs for direct comparison with the domain-adaptation literature.

## Feature Extraction: List A vs. List B

The notebook also extracts the features used by the fold discriminators.

- **List B (Shift features):** The features utilized by the fold discriminators to separate distributions.
- **List A (Activity features):** The activity-model features previously saved by the main OOD-vs-random study.

The List A vs List B overlap is computed fold-aware: for example, the shift features of `F1_vs_F2` are compared with the activity features from outer fold 1, because fold 1 uses `F1` and `F2` as the internal OOD train/validation pair.

## Linking to OOD-vs-Random Protocol

Finally, the notebook links the pairwise shift score to the OOD-vs-random protocol comparison. For each pair/fold, it computes both the difference in validation optimism and the final test benefit of OOD-based model selection. The main scatter plot uses:

$$\text{test gap} = \text{test PR-AUC}_{OOD} - \text{test PR-AUC}_{random}$$

Positive values indicate that OOD-based validation selected a model that performed better on the held-out OOD test fold.


# Pairwise Fold Distribution Shift Analysis for Lo-Hi Hi Datasets


In [11]:
from __future__ import annotations

import sys
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
)
from scipy.stats import hypergeom

# issues should remain visible during the diagnostic analysis
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=".*X does not have valid feature names.*",
)
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Dataset Paths and Output Directories


In [12]:
DATASETS_MAIN = ["drd2", "hiv", "sol"]
DATASETS_DIAGNOSTIC = ["kdr"]
DATASETS = DATASETS_MAIN

TASK = "hi"

DATASET_LABELS = {
    "drd2": "DRD2",
    "hiv": "HIV",
    "sol": "Sol",
    "kdr": "KDR",
}


def find_project_root(start: Path | None = None) -> Path:
    """
    Walk upward until the project root is found.
    """
    if start is None:
        start = Path.cwd().resolve()

    current = start
    while current != current.parent:
        if (
            (current / "data").exists()
            and (current / "utils").exists()
            and (current / "results").exists()
        ):
            return current
        current = current.parent

    raise RuntimeError(
        "Could not find project root containing data/, utils/, and results/."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.fingerprints import compute_fingerprints

OUT_ROOT = PROJECT_ROOT / "results" / "results_classifier_shift_test" / TASK
FIG_ROOT = OUT_ROOT / "figures"

# Root feature cache directory
FEATURE_CACHE_ROOT = PROJECT_ROOT / "features" / "classifier_shift_test" / TASK

for d in (OUT_ROOT, FIG_ROOT, FEATURE_CACHE_ROOT):
    d.mkdir(parents=True, exist_ok=True)

FINGERPRINTS = ["ecfp4", "maccs", "rdkit_desc"]

TOPK_VALUES = [10, 20, 50]

N_CV_SPLITS = 5

print(f"Datasets     : {DATASETS}")
print(f"Task         : {TASK}")
print(f"Project root : {PROJECT_ROOT}")
print(f"Output root  : {OUT_ROOT}")
print(f"Figure root  : {FIG_ROOT}")
print(f"Feature cache: {FEATURE_CACHE_ROOT}")

Datasets     : ['drd2', 'hiv', 'sol']
Task         : hi
Project root : /Users/francescocapria/Desktop/drug-discovery-lohi
Output root  : /Users/francescocapria/Desktop/drug-discovery-lohi/results/results_classifier_shift_test/hi
Figure root  : /Users/francescocapria/Desktop/drug-discovery-lohi/results/results_classifier_shift_test/hi/figures
Feature cache: /Users/francescocapria/Desktop/drug-discovery-lohi/features/classifier_shift_test/hi


## Sanity Check: Input Data and Previous Results


In [13]:
def get_dataset_paths(dataset: str) -> dict:
    """
    Return all input/output paths for one Hi dataset.
    """
    data_dir = PROJECT_ROOT / "data" / TASK / dataset

    binding_results_dir = (
        PROJECT_ROOT
        / "results"
        / "results_ood_vs_random_shuffle"
        / TASK
        / "cross_dataset"
    )

    out_dir = OUT_ROOT / dataset
    fig_dir = out_dir / "figures"
    feat_dir = out_dir / "discriminator_features"
    feature_cache_dir = FEATURE_CACHE_ROOT / dataset

    for d in (out_dir, fig_dir, feat_dir, feature_cache_dir):
        d.mkdir(parents=True, exist_ok=True)

    return {
        "data_dir": data_dir,
        "binding_results_dir": binding_results_dir,
        "out_dir": out_dir,
        "fig_dir": fig_dir,
        "feat_dir": feat_dir,
        "feature_cache_dir": feature_cache_dir,
    }

## Reconstruct Hi Subsets and Pairwise Fold Task


In [14]:
for dataset in DATASETS_MAIN:
    paths = get_dataset_paths(dataset)
    data_dir = paths["data_dir"]

    print(f"\nChecking {dataset}-{TASK}")
    print("Data dir:", data_dir)

    assert data_dir.exists(), f"Missing data directory: {data_dir}"

    for fname in ["test_1.csv", "test_2.csv", "test_3.csv"]:
        path = data_dir / fname
        assert path.exists(), f"Missing file: {path}"

    if not paths["binding_results_dir"].exists():
        print(
            f"Warning: binding results directory not found: {paths['binding_results_dir']}"
        )
        print(
            "Feature overlap with activity models will be skipped unless this directory exists."
        )
    else:
        print("Binding results directory found.")


Checking drd2-hi
Data dir: /Users/francescocapria/Desktop/drug-discovery-lohi/data/hi/drd2
Binding results directory found.

Checking hiv-hi
Data dir: /Users/francescocapria/Desktop/drug-discovery-lohi/data/hi/hiv
Binding results directory found.

Checking sol-hi
Data dir: /Users/francescocapria/Desktop/drug-discovery-lohi/data/hi/sol
Binding results directory found.


In [15]:
SUBSET_FILES = {
    "F1": "test_3.csv",
    "F2": "test_2.csv",
    "F3": "test_1.csv",
}

PAIRS = list(combinations(["F1", "F2", "F3"], 2))


def load_subsets(dataset: str) -> dict[str, pd.DataFrame]:
    paths = get_dataset_paths(dataset)
    data_dir = paths["data_dir"]

    subsets = {}
    for subset_name, filename in SUBSET_FILES.items():
        path = data_dir / filename
        if not path.exists():
            raise FileNotFoundError(f"Missing file: {path}")

        df = pd.read_csv(path).copy()
        df["subset"] = subset_name
        df["source_file"] = filename
        subsets[subset_name] = df

    return subsets


for dataset in DATASETS_MAIN:
    subsets_tmp = load_subsets(dataset)
    print(f"\n{dataset.upper()}-{TASK}")
    for subset_name, df in subsets_tmp.items():
        print(f"  {subset_name} ({SUBSET_FILES[subset_name]}): {len(df)} molecules")
    print(f"  Pairs: {PAIRS}")


DRD2-hi
  F1 (test_3.csv): 1191 molecules
  F2 (test_2.csv): 1194 molecules
  F3 (test_1.csv): 1190 molecules
  Pairs: [('F1', 'F2'), ('F1', 'F3'), ('F2', 'F3')]

HIV-hi
  F1 (test_3.csv): 7848 molecules
  F2 (test_2.csv): 7848 molecules
  F3 (test_1.csv): 7847 molecules
  Pairs: [('F1', 'F2'), ('F1', 'F3'), ('F2', 'F3')]

SOL-hi
  F1 (test_3.csv): 721 molecules
  F2 (test_2.csv): 721 molecules
  F3 (test_1.csv): 721 molecules
  Pairs: [('F1', 'F2'), ('F1', 'F3'), ('F2', 'F3')]


## Featurization and pair dataset builder


In [16]:
def get_smiles_col(df: pd.DataFrame) -> str:
    """Return the SMILES column name."""
    for col in ["smiles", "SMILES", "canonical_smiles"]:
        if col in df.columns:
            return col
    raise ValueError(
        f"No SMILES column found. Available columns: {df.columns.tolist()}"
    )


def get_feature_names(fp_type: str, n_features: int) -> list[str]:
    """Return feature names for each molecular representation."""
    if fp_type == "ecfp4":
        return [f"ecfp4_bit_{i}" for i in range(n_features)]

    if fp_type == "maccs":
        return [f"maccs_key_{i}" for i in range(n_features)]

    if fp_type == "rdkit_desc":
        try:
            from rdkit import Chem
            from rdkit.Chem import Descriptors

            # Match the exact order used in utils.fingerprints.compute_rdkit_descriptors, which
            # relies on Descriptors.CalcMolDescriptors(mol).values().
            ref_mol = Chem.MolFromSmiles("CC")
            calc_desc = Descriptors.CalcMolDescriptors(ref_mol)
            names = list(calc_desc.keys())

            if len(names) != n_features:
                raise ValueError(
                    f"RDKit descriptor name mismatch: got {len(names)} names "
                    f"for {n_features} features."
                )

            desc_list_names = [name for name, _ in Descriptors._descList]

            if names != desc_list_names[: len(names)]:
                warnings.warn(
                    "RDKit descriptor order from CalcMolDescriptors does not exactly "
                    "match Descriptors._descList. Using CalcMolDescriptors order."
                )

            return names

        except Exception as e:
            warnings.warn(
                f"Could not recover RDKit descriptor names reliably: {e}. "
                "Falling back to index-based descriptor names."
            )
            return [f"rdkit_desc_{i}" for i in range(n_features)]

    return [f"{fp_type}_feature_{i}" for i in range(n_features)]


def build_pair_dataset(
    dataset: str,
    subsets: dict[str, pd.DataFrame],
    fold_a: str,
    fold_b: str,
    fp_type: str,
) -> tuple[np.ndarray, np.ndarray, list[str], pd.DataFrame]:
    """
    Build one binary fold-discrimination dataset.

    Label convention:
        fold_a -> 0
        fold_b -> 1

    Invalid SMILES are removed after featurization using the valid_mask returned
    by compute_fingerprints(), so X, y and pair_df remain aligned.
    """
    df_a = subsets[fold_a].copy()
    df_b = subsets[fold_b].copy()

    smiles_col = get_smiles_col(df_a)
    pair_name = f"{fold_a}_vs_{fold_b}"

    df_a["shift_label"] = 0
    df_b["shift_label"] = 1

    pair_df = pd.concat([df_a, df_b], ignore_index=True)
    pair_df["dataset"] = dataset
    pair_df["pair"] = pair_name
    pair_df["fold_a"] = fold_a
    pair_df["fold_b"] = fold_b

    smiles = pair_df[smiles_col].astype(str).tolist()

    cache_path = (
        get_dataset_paths(dataset)["feature_cache_dir"] / f"{pair_name}_{fp_type}.npz"
    )

    X, valid_mask = compute_fingerprints(
        smiles_list=smiles,
        fp_type=fp_type,
        cache_path=cache_path,
    )

    valid_mask = np.asarray(valid_mask, dtype=bool)

    if len(valid_mask) != len(pair_df):
        raise ValueError(
            f"valid_mask length mismatch for {dataset} {pair_name} {fp_type}: "
            f"{len(valid_mask)} vs {len(pair_df)}"
        )

    n_invalid = int((~valid_mask).sum())
    if n_invalid > 0:
        print(
            f"Removed {n_invalid} invalid SMILES from "
            f"{dataset} | {pair_name} | {fp_type}"
        )

    pair_df = pair_df.loc[valid_mask].reset_index(drop=True)
    y = pair_df["shift_label"].to_numpy(dtype=int)

    if X.shape[0] != len(y):
        raise ValueError(
            f"Feature/label mismatch for {dataset} {pair_name} {fp_type}: "
            f"X has {X.shape[0]} rows, y has {len(y)} rows."
        )

    feature_names = get_feature_names(fp_type, X.shape[1])

    return X, y, feature_names, pair_df

## Utility: Suppress RDKit and Featurization Logs


In [17]:
import os
import contextlib
from rdkit import RDLogger

VERBOSE = False


@contextlib.contextmanager
def silence_output():
    """
    Suppress noisy stdout/stderr and RDKit logs during featurization and fitting.
    """
    RDLogger.DisableLog("rdApp.*")
    with open(os.devnull, "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            yield
    RDLogger.EnableLog("rdApp.*")

## High-Capacity In-Sample Discrimination


In [18]:
def make_high_capacity_discriminators(fp_type: str) -> dict:
    """
    High-capacity / weakly-regularized fold discriminators.
    Used only for in-sample maximum separability.
    """
    scale = fp_type == "rdkit_desc"

    dt = DecisionTreeClassifier(
        criterion="gini",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        ccp_alpha=0.0,
        random_state=RANDOM_STATE,
    )

    lr = LogisticRegression(
        C=1e4,
        l1_ratio=0.0,
        solver="saga",
        max_iter=20000,
        random_state=RANDOM_STATE,
    )

    svm = LinearSVC(
        C=1e4,
        max_iter=20000,
        random_state=RANDOM_STATE,
    )

    if scale:
        lr = Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", lr),
            ]
        )
        svm = Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", svm),
            ]
        )

    return {
        "DT": dt,
        "LR": lr,
        "SVM": svm,
    }


def shift_scores_from_balanced_accuracy(bal_acc: float) -> tuple[float, float]:
    """
    Convert balanced accuracy into two equivalent fold-shift scores.

    shift_score_01:
        s = max(0, 2 * balanced_accuracy - 1)

        s = 0 means that the discriminator is not better than chance.
        s = 1 means perfect fold distinguishability.

    proxy_A_distance:
        d_A = 2 * s

        This matches the proxy A-distance definition:
            d_A = 2 * (1 - 2 * error)

        Since balanced_accuracy = 1 - error:
            d_A = 2 * (2 * balanced_accuracy - 1)

    We clip at zero because balanced accuracy below 0.5 means the discriminator
    is not distinguishable above chance in the chosen orientation.
    """
    if not np.isfinite(bal_acc):
        return np.nan, np.nan

    shift_score_01 = max(0.0, 2.0 * float(bal_acc) - 1.0)
    proxy_a_distance = 2.0 * shift_score_01

    return shift_score_01, proxy_a_distance


insample_rows = []
fitted_high_capacity = {}

for dataset in DATASETS_MAIN:
    subsets = load_subsets(dataset)

    if VERBOSE:
        print(
            f"\n=== High-capacity in-sample discrimination: {dataset.upper()}-{TASK} ==="
        )

    for fold_a, fold_b in PAIRS:
        pair_name = f"{fold_a}_vs_{fold_b}"

        for fp_type in FINGERPRINTS:
            try:
                with silence_output():
                    X, y, feature_names, pair_df = build_pair_dataset(
                        dataset=dataset,
                        subsets=subsets,
                        fold_a=fold_a,
                        fold_b=fold_b,
                        fp_type=fp_type,
                    )
            except Exception as e:
                print(f"Skip {dataset} | {pair_name} | {fp_type}: {e}")
                continue

            for model_name, model in make_high_capacity_discriminators(fp_type).items():
                if VERBOSE:
                    print(f"{dataset} | {pair_name} | {fp_type} | {model_name}")

                fitted = clone(model)

                with silence_output():
                    fitted.fit(X, y)

                y_pred = fitted.predict(X)

                acc = accuracy_score(y, y_pred)
                bal_acc = balanced_accuracy_score(y, y_pred)
                shift_score_01, proxy_a_distance = shift_scores_from_balanced_accuracy(
                    bal_acc
                )

                insample_rows.append(
                    {
                        "dataset": dataset,
                        "task": TASK,
                        "pair": pair_name,
                        "fold_a": fold_a,
                        "fold_b": fold_b,
                        "fingerprint": fp_type,
                        "model": model_name,
                        "evaluation": "high_capacity_insample",
                        "accuracy": acc,
                        "balanced_accuracy": bal_acc,
                        "shift_score_01": shift_score_01,
                        "proxy_A_distance": proxy_a_distance,
                        "n_samples": len(y),
                        "n_class_0": int((y == 0).sum()),
                        "n_class_1": int((y == 1).sum()),
                        "main_shift_estimate": False,
                    }
                )

                fitted_high_capacity[(dataset, pair_name, fp_type, model_name)] = {
                    "model": fitted,
                    "feature_names": feature_names,
                    "X": X,
                    "y": y,
                    "pair_df": pair_df,
                }

df_insample = pd.DataFrame(insample_rows)

for dataset, sub in df_insample.groupby("dataset"):
    out_path = (
        get_dataset_paths(dataset)["out_dir"]
        / "discrimination_high_capacity_insample.csv"
    )
    sub.to_csv(out_path, index=False)

df_insample.to_csv(
    OUT_ROOT / "cross_dataset_discrimination_high_capacity_insample.csv",
    index=False,
)

display(
    df_insample.pivot_table(
        index=["dataset", "pair", "fingerprint"],
        columns="model",
        values="balanced_accuracy",
    ).round(3)
)

model                            DT     LR    SVM
dataset pair     fingerprint                     
drd2    F1_vs_F2 ecfp4        1.000  1.000  1.000
                 maccs        1.000  0.957  0.952
                 rdkit_desc   1.000  0.975  0.983
        F1_vs_F3 ecfp4        1.000  1.000  1.000
                 maccs        1.000  0.932  0.927
                 rdkit_desc   1.000  0.951  0.963
        F2_vs_F3 ecfp4        1.000  1.000  1.000
                 maccs        1.000  0.974  0.972
                 rdkit_desc   1.000  0.977  0.985
hiv     F1_vs_F2 ecfp4        1.000  0.847  0.845
                 maccs        1.000  0.690  0.692
                 rdkit_desc   1.000  0.700  0.701
        F1_vs_F3 ecfp4        1.000  0.828  0.828
                 maccs        1.000  0.688  0.690
                 rdkit_desc   1.000  0.706  0.707
        F2_vs_F3 ecfp4        1.000  0.872  0.871
                 maccs        0.999  0.734  0.735
                 rdkit_desc   1.000  0.750  0.751
sol     F1_vs_F2 ecfp4        1.000  1.000  1.000
                 maccs        1.000  0.651  0.655
                 rdkit_desc   1.000  0.676  0.691
        F1_vs_F3 ecfp4        1.000  1.000  1.000
                 maccs        0.999  0.714  0.716
                 rdkit_desc   1.000  0.724  0.734
        F2_vs_F3 ecfp4        1.000  1.000  1.000
                 maccs        1.000  0.693  0.692
                 rdkit_desc   1.000  0.707  0.712

## Same-Search Out-of-Sample Discrimination


In [21]:
# ============================================================
# Classifier-shift / List B generation
# SAME SEARCH GRIDS AS STANDARD MODELS
#
# Outputs:
#   results/results_classifier_shift_test/hi/cross_dataset_discrimination_same_search_cv.csv
#   results/results_classifier_shift_test/hi/cross_dataset_listB_same_search_cv_feature_importance.csv
# ============================================================

from __future__ import annotations

import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

# ------------------------------------------------------------
# 0. Global config
# ------------------------------------------------------------

RANDOM_STATE = 42
N_BITS = 1024
RADIUS = 2
N_JOBS = -1

# For fold-discrimination, accuracy is the natural score.
SCORING = "balanced_accuracy"

# Set False if you only want to reuse existing outputs.
FORCE_RERUN = True

DATASETS = ["drd2", "hiv", "sol"]
FOLD_PAIRS = [("F1", "F2"), ("F1", "F3"), ("F2", "F3")]

# Lo-Hi mapping:
# train_1 = F1 + F2, test_1 = F3
# train_2 = F1 + F3, test_2 = F2
# train_3 = F2 + F3, test_3 = F1
SUBSET_FILES = {
    "F1": "test_3.csv",
    "F2": "test_2.csv",
    "F3": "test_1.csv",
}

warnings.filterwarnings(
    "ignore",
    message=".*'penalty' was deprecated.*",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    message=".*The max_iter was reached.*",
    category=UserWarning,
)


# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------


def find_project_root() -> Path:
    candidates = [
        Path.home() / "Desktop" / "drug-discovery-lohi",
        Path.home() / "drug-discovery-lohi",
        Path.cwd(),
    ]

    for c in candidates:
        if c.exists() and (c / "data").exists():
            return c.resolve()

    current = Path.cwd().resolve()
    while current != current.parent:
        if (current / "data").exists():
            return current
        current = current.parent

    raise RuntimeError("Could not find project root containing data/.")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

TASK = "hi"
DATA_DIR = PROJECT_ROOT / "data" / TASK
OUT_DIR = PROJECT_ROOT / "results" / "results_classifier_shift_test" / TASK
OUT_DIR.mkdir(parents=True, exist_ok=True)

DISC_OUT = OUT_DIR / "cross_dataset_discrimination_same_search_cv.csv"
LISTB_OUT = OUT_DIR / "cross_dataset_listB_same_search_cv_feature_importance.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("N_BITS:", N_BITS)


# ------------------------------------------------------------
# 2. Fingerprint utilities
# ------------------------------------------------------------


def detect_smiles_column(df: pd.DataFrame) -> str:
    candidates = [
        "smiles",
        "SMILES",
        "canonical_smiles",
        "canonical_smiles_rdkit",
        "mol",
        "molecule",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(f"No SMILES column found. Columns: {list(df.columns)}")


def compute_ecfp4_1024(smiles_list: list[str]) -> tuple[np.ndarray, np.ndarray]:
    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=RADIUS,
        fpSize=N_BITS,
    )

    fps = []
    valid_mask = []

    for smi in smiles_list:
        mol = Chem.MolFromSmiles(str(smi))

        if mol is None:
            fps.append(np.zeros(N_BITS, dtype=np.uint8))
            valid_mask.append(False)
            continue

        bv = generator.GetFingerprint(mol)
        arr = np.zeros((N_BITS,), dtype=np.uint8)
        DataStructs.ConvertToNumpyArray(bv, arr)

        fps.append(arr)
        valid_mask.append(True)

    X = np.vstack(fps).astype(np.uint8)
    valid_mask = np.asarray(valid_mask, dtype=bool)

    return X, valid_mask


def load_lohi_subsets(dataset: str) -> dict[str, pd.DataFrame]:
    dataset_dir = DATA_DIR / dataset

    if not dataset_dir.exists():
        raise FileNotFoundError(f"Dataset directory not found: {dataset_dir}")

    subsets = {}

    for fold_name, filename in SUBSET_FILES.items():
        path = dataset_dir / filename

        if not path.exists():
            raise FileNotFoundError(f"Missing Lo-Hi subset file: {path}")

        df = pd.read_csv(path)
        smiles_col = detect_smiles_column(df)

        df = df.copy()
        df["dataset"] = dataset
        df["fold_origin"] = fold_name
        df["smiles_for_fp"] = df[smiles_col].astype(str)

        subsets[fold_name] = df

    return subsets


def featurize_subsets(subsets: dict[str, pd.DataFrame]) -> dict[str, dict]:
    out = {}

    for fold_name, df in subsets.items():
        X, valid_mask = compute_ecfp4_1024(df["smiles_for_fp"].tolist())

        if not valid_mask.all():
            n_bad = int((~valid_mask).sum())
            print(f"Warning: dropping {n_bad} invalid SMILES in {fold_name}")
            df = df.loc[valid_mask].reset_index(drop=True)
            X = X[valid_mask]

        out[fold_name] = {
            "df": df.reset_index(drop=True),
            "X": X,
        }

    return out


# ------------------------------------------------------------
# 3. Exact same search grids as standard models
# ------------------------------------------------------------

DT_GRID = {
    "max_depth": [3, 5, 8, 10, 15, 20, 30, None],
    "min_samples_split": [2, 5, 10, 20, 50],
    "min_samples_leaf": [1, 2, 5, 10, 20, 50],
    "criterion": ["gini", "entropy"],
    "class_weight": [None, "balanced"],
    "max_features": ["sqrt", "log2", None],
    "ccp_alpha": [0.0, 1e-05, 0.0001, 0.001, 0.01],
}

LR_GRID_DRD2_SOL = {
    "C": [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 100.0],
    "l1_ratio": [0.0, 0.25, 0.5, 0.75, 1.0],
    "class_weight": [None, "balanced"],
}

LR_GRID_HIV = {
    "C": [0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 5.0],
    "l1_ratio": [0.0, 0.25, 0.5, 0.9, 1.0],
    "class_weight": [None, "balanced"],
}

SVM_LINEAR_GRID = {
    "C": [
        0.0001,
        0.0005,
        0.001,
        0.002,
        0.005,
        0.01,
        0.05,
        0.1,
        0.25,
        0.5,
        1.0,
        2.0,
        5.0,
        10.0,
        25.0,
        50.0,
        100.0,
    ],
    "class_weight": [None, "balanced"],
}


def get_model_specs(dataset: str) -> dict:
    if dataset == "hiv":
        lr_max_iter = 60000
        lr_grid = LR_GRID_HIV
    else:
        lr_max_iter = 15000
        lr_grid = LR_GRID_DRD2_SOL

    return {
        "DT": {
            "model_label": "Decision Tree",
            "estimator": DecisionTreeClassifier(
                random_state=RANDOM_STATE,
            ),
            "param_grid": DT_GRID,
        },
        "LR": {
            "model_label": "Logistic Regression",
            "estimator": LogisticRegression(
                solver="saga",
                penalty="elasticnet",
                max_iter=lr_max_iter,
                tol=0.001,
                random_state=RANDOM_STATE,
            ),
            "param_grid": lr_grid,
        },
        "SVM": {
            "model_label": "Linear SVM",
            "estimator": SVC(
                kernel="linear",
                probability=False,
            ),
            "param_grid": SVM_LINEAR_GRID,
        },
    }


# ------------------------------------------------------------
# 4. Sanity check grid sizes
# ------------------------------------------------------------


def grid_size(grid: dict) -> int:
    n = 1
    for values in grid.values():
        n *= len(values)
    return n


print("\nGrid size check:")
print("DT:", grid_size(DT_GRID), "expected 14400")
print("LR DRD2/SOL:", grid_size(LR_GRID_DRD2_SOL), "expected 110")
print("LR HIV:", grid_size(LR_GRID_HIV), "expected 70")
print("SVM:", grid_size(SVM_LINEAR_GRID), "expected 34")


# ------------------------------------------------------------
# 5. Feature-importance extraction
# ------------------------------------------------------------


def tree_minimum_depths(tree_model, n_features: int) -> dict[int, int]:
    tree = tree_model.tree_

    feature = tree.feature
    children_left = tree.children_left
    children_right = tree.children_right

    min_depth = {}
    stack = [(0, 0)]

    while stack:
        node_id, depth = stack.pop()
        feat_idx = int(feature[node_id])

        if feat_idx >= 0:
            if feat_idx not in min_depth:
                min_depth[feat_idx] = depth
            else:
                min_depth[feat_idx] = min(min_depth[feat_idx], depth)

            left = children_left[node_id]
            right = children_right[node_id]

            if left != -1:
                stack.append((left, depth + 1))
            if right != -1:
                stack.append((right, depth + 1))

    return min_depth


def extract_feature_importance(
    estimator, model_name: str, n_features: int = N_BITS
) -> pd.DataFrame:
    if model_name == "DT":
        importance = np.asarray(estimator.feature_importances_, dtype=float)
        signed_weight = np.full(n_features, np.nan)

        min_depths = tree_minimum_depths(estimator, n_features)
        tree_min_depth = np.array(
            [min_depths.get(i, np.nan) for i in range(n_features)],
            dtype=float,
        )
        used_in_tree = np.array(
            [i in min_depths for i in range(n_features)],
            dtype=bool,
        )

    elif model_name in {"LR", "SVM"}:
        coef = np.asarray(estimator.coef_).ravel().astype(float)

        importance = np.abs(coef)
        signed_weight = coef
        tree_min_depth = np.full(n_features, np.nan)
        used_in_tree = np.full(n_features, False)

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    out = pd.DataFrame(
        {
            "feature_idx": np.arange(n_features, dtype=int),
            "feature_name": [f"ecfp4_bit_{i}" for i in range(n_features)],
            "importance": importance,
            "signed_weight": signed_weight,
            "tree_min_depth": tree_min_depth,
            "used_in_tree": used_in_tree,
        }
    )

    # Compatibility columns for downstream Tanimoto List B loading.
    if model_name == "DT":
        out["tree_importance"] = out["importance"]
        out["coefficient"] = np.nan
        out["abs_importance"] = out["importance"]
        out["importance_source"] = "tree_importance"
    else:
        out["tree_importance"] = np.nan
        out["coefficient"] = out["signed_weight"]
        out["abs_importance"] = out["importance"]
        out["importance_source"] = "abs_coefficient"

    out["importance_value"] = out["importance"]

    importance_sum = float(out["importance"].sum())

    if importance_sum > 0:
        out["normalized_importance"] = out["importance"] / importance_sum
    else:
        out["normalized_importance"] = 0.0

    out = out.sort_values(
        ["importance", "feature_idx"],
        ascending=[False, True],
    ).reset_index(drop=True)

    out["rank"] = np.arange(1, len(out) + 1, dtype=int)

    out = out[out["importance"] > 0].copy()

    return out


# ------------------------------------------------------------
# 6. CV helpers
# ------------------------------------------------------------


def make_cv(y: np.ndarray, max_splits: int = 5) -> StratifiedKFold:
    counts = pd.Series(y).value_counts()
    min_count = int(counts.min())
    n_splits = min(max_splits, min_count)

    if n_splits < 2:
        raise ValueError(
            f"Cannot build StratifiedKFold. Class counts: {counts.to_dict()}"
        )

    return StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )


def proxy_a_distance_from_balanced_accuracy(bal_acc: float) -> float:
    return float(np.clip(4.0 * bal_acc - 2.0, 0.0, 2.0))


# ------------------------------------------------------------
# 7. Main run
# ------------------------------------------------------------

if FORCE_RERUN:
    for p in [DISC_OUT, LISTB_OUT]:
        if p.exists():
            print(f"Removing old output: {p}")
            p.unlink()

disc_rows = []
listb_rows = []

t0_all = time.time()

for dataset in DATASETS:
    print("\n" + "=" * 100)
    print(f"Dataset: {dataset}")
    print("=" * 100)

    subsets = load_lohi_subsets(dataset)
    feat = featurize_subsets(subsets)
    specs = get_model_specs(dataset)

    for fold_a, fold_b in FOLD_PAIRS:
        pair_name = f"{fold_a}_vs_{fold_b}"

        Xa = feat[fold_a]["X"]
        Xb = feat[fold_b]["X"]

        X = np.vstack([Xa, Xb]).astype(np.uint8)
        y = np.array([0] * len(Xa) + [1] * len(Xb), dtype=int)

        cv = make_cv(y, max_splits=5)

        print(
            f"\nPair: {pair_name} | "
            f"X={X.shape} | "
            f"class counts={dict(pd.Series(y).value_counts())}"
        )

        for model_name, spec in specs.items():
            model_label = spec["model_label"]
            estimator = spec["estimator"]
            param_grid = spec["param_grid"]

            print(
                f"  Running {model_name} ({model_label}) | grid={grid_size(param_grid)} configs"
            )

            search = GridSearchCV(
                estimator=estimator,
                param_grid=param_grid,
                scoring=SCORING,
                cv=cv,
                refit=True,
                n_jobs=N_JOBS,
                return_train_score=True,
                error_score="raise",
                verbose=0,
            )

            t0 = time.time()
            search.fit(X, y)
            elapsed = time.time() - t0

            best_est = search.best_estimator_
            best_cv_acc = float(search.best_score_)

            cv_results = pd.DataFrame(search.cv_results_)
            best_idx = int(search.best_index_)
            best_cv_std = float(cv_results.loc[best_idx, "std_test_score"])

            y_pred_full = best_est.predict(X)
            full_acc = float(accuracy_score(y, y_pred_full))
            full_bal_acc = float(balanced_accuracy_score(y, y_pred_full))

            full_auc = np.nan
            if hasattr(best_est, "decision_function"):
                try:
                    score_full = best_est.decision_function(X)
                    full_auc = float(roc_auc_score(y, score_full))
                except Exception:
                    full_auc = np.nan
            elif hasattr(best_est, "predict_proba"):
                try:
                    score_full = best_est.predict_proba(X)[:, 1]
                    full_auc = float(roc_auc_score(y, score_full))
                except Exception:
                    full_auc = np.nan

            best_params_json = json.dumps(search.best_params_, sort_keys=True)

            disc_rows.append(
                {
                    "dataset": dataset,
                    "fp_type": "ecfp4",
                    "n_bits": N_BITS,
                    "pair": pair_name,
                    "fold_a": fold_a,
                    "fold_b": fold_b,
                    "model": model_name,
                    "model_label": model_label,
                    "search_name": "same_search_cv",
                    "scoring": SCORING,
                    "n_samples": int(X.shape[0]),
                    "n_features": int(X.shape[1]),
                    "n_fold_a": int(len(Xa)),
                    "n_fold_b": int(len(Xb)),
                    "cv_n_splits": int(cv.get_n_splits()),
                    "cv_accuracy_mean": best_cv_acc,
                    "cv_accuracy_std": best_cv_std,
                    "proxy_a_distance": proxy_a_distance_from_accuracy(best_cv_acc),
                    "full_fit_accuracy": full_acc,
                    "full_fit_balanced_accuracy": full_bal_acc,
                    "full_fit_roc_auc": full_auc,
                    "best_params": best_params_json,
                    "grid_size": int(grid_size(param_grid)),
                    "elapsed_sec": float(elapsed),
                }
            )

            fi = extract_feature_importance(
                estimator=best_est,
                model_name=model_name,
                n_features=N_BITS,
            )

            fi.insert(0, "dataset", dataset)
            fi.insert(1, "fp_type", "ecfp4")
            fi.insert(2, "n_bits", N_BITS)
            fi.insert(3, "pair", pair_name)
            fi.insert(4, "fold_a", fold_a)
            fi.insert(5, "fold_b", fold_b)
            fi.insert(6, "model", model_name)
            fi.insert(7, "model_label", model_label)
            fi.insert(8, "search_name", "same_search_cv")
            fi["best_params"] = best_params_json

            listb_rows.append(fi)

            print(
                f"    best_cv_acc={best_cv_acc:.4f} | "
                f"std={best_cv_std:.4f} | "
                f"A-dist={proxy_a_distance_from_balanced_accuracy(best_cv_acc):.4f} | "
                f"nonzero_features={len(fi)} | "
                f"time={elapsed / 60:.1f} min"
            )


# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------

disc_df = pd.DataFrame(disc_rows)
listb_df = pd.concat(listb_rows, ignore_index=True) if listb_rows else pd.DataFrame()

disc_df.to_csv(DISC_OUT, index=False)
listb_df.to_csv(LISTB_OUT, index=False)

print("\n" + "=" * 100)
print("DONE")
print("=" * 100)

print(f"Saved discrimination CSV:\n  {DISC_OUT}")
print(f"shape: {disc_df.shape}")

print(f"\nSaved List B feature-importance CSV:\n  {LISTB_OUT}")
print(f"shape: {listb_df.shape}")

print(f"\nTotal time: {(time.time() - t0_all) / 60:.1f} min")

print("\nDiscrimination preview:")
display(disc_df.head())

print("\nList B preview:")
display(listb_df.head())

print("\nFinal file check:")
for p in [DISC_OUT, LISTB_OUT]:
    print(f"{p} | exists={p.exists()} | size={p.stat().st_size / 1024:.1f} KB")

PROJECT_ROOT: /Users/francescocapria/Desktop/drug-discovery-lohi
DATA_DIR: /Users/francescocapria/Desktop/drug-discovery-lohi/data/hi
OUT_DIR: /Users/francescocapria/Desktop/drug-discovery-lohi/results/results_classifier_shift_test/hi
N_BITS: 1024

Grid size check:
DT: 14400 expected 14400
LR DRD2/SOL: 110 expected 110
LR HIV: 70 expected 70
SVM: 34 expected 34

Dataset: drd2

Pair: F1_vs_F2 | X=(2385, 1024) | class counts={1: np.int64(1194), 0: np.int64(1191)}
  Running DT (Decision Tree) | grid=14400 configs


KeyboardInterrupt: 

## Extract List B: Fold-Discriminator Feature Importances

### Which molecular features enable the discriminator to determine whether a molecule originates from F1, F2 or F3?


In [ ]:
def get_base_model(model):
    return model.steps[-1][1] if isinstance(model, Pipeline) else model


def tree_minimum_depths(tree_model, n_features: int) -> np.ndarray:
    tree = tree_model.tree_
    min_depth = np.full(n_features, np.nan)
    stack = [(0, 0)]
    while stack:
        node_id, depth = stack.pop()
        feature_id = tree.feature[node_id]
        if feature_id >= 0:
            if np.isnan(min_depth[feature_id]) or depth < min_depth[feature_id]:
                min_depth[feature_id] = depth
            left = tree.children_left[node_id]
            right = tree.children_right[node_id]
            if left != -1:
                stack.append((left, depth + 1))
            if right != -1:
                stack.append((right, depth + 1))
    return min_depth


from sklearn.inspection import permutation_importance


def extract_discriminator_importance(
    fitted_model,
    feature_names: list[str],
    model_name: str,
    X_eval=None,
    y_eval=None,
    perm_n_repeats: int = 10,
    perm_scoring: str = "balanced_accuracy",
    perm_n_jobs: int = -1,
    random_state: int = RANDOM_STATE,
    perm_eval_set_label: str = "fold_discriminator_dataset",
) -> pd.DataFrame:
    """
    Extract feature importances from a fitted fold discriminator.

    DT     -> tree_importance as the main ranking signal
    LR/SVM -> absolute coefficient

    Permutation importance for DT can still be saved as a diagnostic column
    when X_eval/y_eval are provided, but it is NOT used as the main List B ranking.
    """
    base = get_base_model(fitted_model)
    n_features = len(feature_names)

    if model_name == "DT":
        tree_importance = np.asarray(base.feature_importances_, dtype=float)
        min_depth = tree_minimum_depths(base, n_features)

        df = pd.DataFrame(
            {
                "feature_idx": np.arange(n_features),
                "feature_name": feature_names,
                "tree_importance": tree_importance,
                "minimum_depth": min_depth,
                "used": tree_importance > 0,
            }
        )

        tree_total = df["tree_importance"].sum()
        df["normalized_tree_importance"] = (
            df["tree_importance"] / tree_total if tree_total > 0 else 0.0
        )

        df["rank_tree_importance"] = (
            df["tree_importance"].rank(ascending=False, method="first").astype(int)
        )

        # Optional diagnostic only: permutation importance on held-out data.
        # It is saved, but NOT used as the main ranking.
        if X_eval is not None and y_eval is not None:
            perm = permutation_importance(
                fitted_model,
                X_eval,
                y_eval,
                scoring=perm_scoring,
                n_repeats=perm_n_repeats,
                random_state=random_state,
                n_jobs=perm_n_jobs,
            )

            perm_df = pd.DataFrame(
                {
                    "feature_idx": np.arange(len(perm.importances_mean), dtype=int),
                    "permutation_importance_mean": perm.importances_mean.astype(float),
                    "permutation_importance_std": perm.importances_std.astype(float),
                }
            )
            perm_df["permutation_scoring"] = perm_scoring
            perm_df["permutation_eval_set"] = perm_eval_set_label
            perm_df["permutation_n_repeats"] = int(perm_n_repeats)

            df = df.merge(perm_df, on="feature_idx", how="left")
        else:
            df["permutation_importance_mean"] = np.nan
            df["permutation_importance_std"] = np.nan
            df["permutation_scoring"] = np.nan
            df["permutation_eval_set"] = np.nan
            df["permutation_n_repeats"] = np.nan

        # Main List B ranking for DT.
        df["importance"] = df["tree_importance"]
        df["abs_importance"] = df["tree_importance"].clip(lower=0.0)
        df["importance_source"] = "tree_importance"
        df["coefficient"] = np.nan
        df["direction"] = "tree"

        df = df.sort_values(
            ["tree_importance", "minimum_depth", "feature_idx"],
            ascending=[False, True, True],
            na_position="last",
        ).reset_index(drop=True)

        df["rank"] = np.arange(1, len(df) + 1)

    else:
        coef = np.asarray(base.coef_).ravel()
        importance = np.abs(coef)

        df = pd.DataFrame(
            {
                "feature_idx": np.arange(len(coef)),
                "feature_name": feature_names[: len(coef)],
                "importance": importance,
                "abs_importance": importance,
                "coefficient": coef,
                "direction": np.where(
                    coef > 0,
                    "towards_fold_b",
                    np.where(coef < 0, "towards_fold_a", "zero"),
                ),
                "tree_importance": np.nan,
                "normalized_tree_importance": np.nan,
                "rank_tree_importance": np.nan,
                "minimum_depth": np.nan,
                "used": importance > 0,
                "importance_source": "absolute_coefficient",
            }
        )

        df = df.sort_values(
            ["abs_importance", "feature_idx"],
            ascending=[False, True],
        ).reset_index(drop=True)

        df["rank"] = np.arange(1, len(df) + 1)

    total = df["abs_importance"].sum()
    df["normalized_importance"] = df["abs_importance"] / total if total > 0 else 0.0

    return df


def collect_listB_from_models(model_dict: dict, source_name: str) -> pd.DataFrame:
    """
    Extract List B from a fitted-model dictionary.
    """
    if source_name == "same_search_cv":
        perm_eval_set_label = "fold_discriminator_holdout"
    elif source_name == "high_capacity_insample":
        perm_eval_set_label = "fold_discriminator_insample"
    else:
        perm_eval_set_label = "fold_discriminator_dataset"

    all_rows = []

    for (dataset, pair_name, fp_type, model_name), obj in model_dict.items():
        fitted_model = obj["model"]
        feature_names = obj["feature_names"]
        X_eval = obj.get("X", None)
        y_eval = obj.get("y", None)

        df_imp = extract_discriminator_importance(
            fitted_model=fitted_model,
            feature_names=feature_names,
            model_name=model_name,
            X_eval=X_eval,
            y_eval=y_eval,
            perm_n_repeats=10,
            perm_scoring="balanced_accuracy",
            perm_n_jobs=-1,
            random_state=RANDOM_STATE,
            perm_eval_set_label=perm_eval_set_label,
        )

        fold_a, fold_b = pair_name.split("_vs_")
        df_imp.insert(0, "source", source_name)
        df_imp.insert(0, "model", model_name)
        df_imp.insert(0, "fingerprint", fp_type)
        df_imp.insert(0, "fold_b", fold_b)
        df_imp.insert(0, "fold_a", fold_a)
        df_imp.insert(0, "pair", pair_name)
        df_imp.insert(0, "task", TASK)
        df_imp.insert(0, "dataset", dataset)

        all_rows.append(df_imp)

        top = df_imp.head(50)
        feat_dir = get_dataset_paths(dataset)["feat_dir"]
        top.to_csv(
            feat_dir / f"listB_{source_name}_{pair_name}_{fp_type}_{model_name}.csv",
            index=False,
        )

    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()


listB_parts = []

if "fitted_high_capacity" in globals() and fitted_high_capacity:
    listB_parts.append(
        collect_listB_from_models(
            model_dict=fitted_high_capacity,
            source_name="high_capacity_insample",
        )
    )

if "best_search_models" in globals() and best_search_models:
    listB_parts.append(
        collect_listB_from_models(
            model_dict=best_search_models,
            source_name="same_search_cv",
        )
    )

df_listB = pd.concat(listB_parts, ignore_index=True)

for dataset, sub in df_listB.groupby("dataset"):
    sub.to_csv(
        get_dataset_paths(dataset)["out_dir"]
        / "listB_discriminator_feature_importance.csv",
        index=False,
    )

df_listB.to_csv(
    OUT_ROOT / "cross_dataset_listB_discriminator_feature_importance.csv",
    index=False,
)

df_listB_main = df_listB[df_listB["source"] == "same_search_cv"].copy()
df_listB_insample = df_listB[df_listB["source"] == "high_capacity_insample"].copy()

df_listB_main.to_csv(
    OUT_ROOT / "cross_dataset_listB_same_search_cv_feature_importance.csv",
    index=False,
)
df_listB_insample.to_csv(
    OUT_ROOT / "cross_dataset_listB_high_capacity_insample_feature_importance.csv",
    index=False,
)

source_check = (
    df_listB_main.groupby(["model", "importance_source"], as_index=False)
    .size()
    .rename(columns={"size": "n_features"})
)
display(source_check)

dt_bad = source_check[
    (source_check["model"] == "DT")
    & (source_check["importance_source"] != "tree_importance")
]
assert len(dt_bad) == 0, (
    "Some DT List B rankings did not use tree_importance:\n"
    f"{dt_bad.to_string(index=False)}"
)

linear_bad = source_check[
    (source_check["model"].isin(["LR", "SVM"]))
    & (source_check["importance_source"] != "absolute_coefficient")
]
assert len(linear_bad) == 0, (
    "Some LR/SVM List B rankings did not use absolute_coefficient:\n"
    f"{linear_bad.to_string(index=False)}"
)

print("OK: List B importance sources are aligned with final ranking conventions.")
print("OK: DT List B uses tree_importance as the main ranking signal.")
print(
    "Saved List B feature-importance tables:\n"
    f"  full mixed-source table: {len(df_listB)} rows\n"
    f"  same_search_cv only    : {len(df_listB_main)} rows\n"
    f"  high_capacity only     : {len(df_listB_insample)} rows\n"
    "Use same_search_cv for the main List A/List B overlap analysis."
)

display(
    df_listB.query("rank <= 10")
    .loc[
        :,
        [
            "dataset",
            "source",
            "pair",
            "fingerprint",
            "model",
            "rank",
            "feature_idx",
            "feature_name",
            "abs_importance",
            "coefficient",
            "direction",
            "minimum_depth",
        ],
    ]
    .head(30)
)

KeyboardInterrupt: 

## Load List A: Activity-Model Feature Importances


In [ ]:
OOD_CROSS_DIR = (
    PROJECT_ROOT / "results" / "results_ood_vs_random_shuffle" / TASK / "cross_dataset"
)

# Main diagnostic List A:
# use the tree-based activity feature table when available.
# This means:
#   DT     -> tree_importance
#   LR/SVM -> coefficient-based importance
#
# The permutation-based version remains available as sensitivity analysis:
#   cross_dataset_feature_importance_all_permutation.csv

LISTA_IMPORTANCE_MODE = "tree"

LISTA_PATH = (
    OOD_CROSS_DIR / f"cross_dataset_feature_importance_all_{LISTA_IMPORTANCE_MODE}.csv"
)

if not LISTA_PATH.exists():
    print(
        f"WARNING: {LISTA_PATH.name} not found. "
        "Falling back to unsuffixed cross_dataset_feature_importance_all.csv"
    )
    LISTA_PATH = OOD_CROSS_DIR / "cross_dataset_feature_importance_all.csv"

if not LISTA_PATH.exists():
    raise FileNotFoundError(
        f"Missing List A table from OOD-vs-random study: {LISTA_PATH}"
    )

df_listA = pd.read_csv(LISTA_PATH, low_memory=False).copy()

print(f"Loaded List A from: {LISTA_PATH.name}")

# Keep only datasets used in the valid protocol comparison.
df_listA = df_listA[df_listA["dataset"].isin(DATASETS_MAIN)].copy()

# Normalize model names.
MODEL_NAME_MAP = {
    "Decision Tree": "DT",
    "Logistic Regression": "LR",
    "Linear SVM": "SVM",
    "dt": "DT",
    "decision_tree": "DT",
    "lr": "LR",
    "logistic_regression": "LR",
    "svm": "SVM",
    "svm_linear": "SVM",
}

df_listA["model"] = (
    df_listA["model"].map(MODEL_NAME_MAP).fillna(df_listA["model"].astype(str))
)

# Normalize fingerprint names to match List B.
FP_MAP = {
    "ECFP4": "ecfp4",
    "MACCS": "maccs",
    "RDKit desc": "rdkit_desc",
    "ecfp4": "ecfp4",
    "maccs": "maccs",
    "rdkit_desc": "rdkit_desc",
}

df_listA["fingerprint"] = (
    df_listA["fingerprint"].map(FP_MAP).fillna(df_listA["fingerprint"].astype(str))
)

if "importance_value" not in df_listA.columns:
    raise ValueError(
        "Expected 'importance_value' in the List A table. "
        "Re-run the OOD-vs-random table-building notebook first."
    )

df_listA["importance_value_numeric"] = pd.to_numeric(
    df_listA["importance_value"], errors="coerce"
).fillna(0.0)

# Main activity importance used for List A.
# In tree mode, DT importance_value is tree_importance and is already non-negative.
# For linear models, importance_value is coefficient-based and already non-negative
# if normalized_abs_importance was used; abs() is kept for robustness.
df_listA["activity_importance"] = np.where(
    df_listA["model"].eq("DT"),
    df_listA["importance_value_numeric"].clip(lower=0.0),
    df_listA["importance_value_numeric"].abs(),
)

sort_cols = ["dataset", "model", "fingerprint", "protocol", "fold"]

df_listA = df_listA.sort_values(
    sort_cols + ["activity_importance", "feature_idx"],
    ascending=[True] * len(sort_cols) + [False, True],
).reset_index(drop=True)

df_listA["activity_rank"] = (
    df_listA.groupby(sort_cols)["activity_importance"]
    .rank(method="first", ascending=False)
    .astype(int)
)

df_listA.to_csv(
    OUT_ROOT / "cross_dataset_listA_activity_feature_importance.csv",
    index=False,
)

display(df_listA.query("activity_rank <= 10").head(10))

Loaded List A from: cross_dataset_feature_importance_all_tree.csv


,feature_idx,feature_name,tree_importance,minimum_depth,used_in_tree,normalized_tree_importance,rank_tree_importance,permutation_importance_mean,permutation_importance_std,permutation_scoring,...,fold,dt_importance_mode,raw_weight,abs_weight,normalized_abs_importance,direction,rank_abs_weight,importance_value_numeric,activity_importance,activity_rank
0,2000,ecfp4_bit_2000,0.095092,0.0,True,0.095092,1.0,0.007348,0.005241,average_precision,...,1,tree,NaN,NaN,NaN,NaN,NaN,0.095092,0.095092,1
1,1145,ecfp4_bit_1145,0.051899,5.0,True,0.051899,2.0,0.034229,0.004964,average_precision,...,1,tree,NaN,NaN,NaN,NaN,NaN,0.051899,0.051899,2
2,1503,ecfp4_bit_1503,0.039868,7.0,True,0.039868,3.0,-0.002270,0.001541,average_precision,...,1,tree,NaN,NaN,NaN,NaN,NaN,0.039868,0.039868,3
3,562,ecfp4_bit_562,0.032088,2.0,True,0.032088,4.0,-0.013747,0.003835,average_precision,...,1,tree,NaN,NaN,NaN,NaN,NaN,0.032088,0.032088,4
4,268,ecfp4_bit_268,0.031189,6.0,True,0.031189,5.0,0.003997,0.001195,average_precision,...,1,tree,NaN,NaN,NaN,NaN,NaN,0.031189,0.031189,5
5,521,ecfp4_bit_521,0.030268,1.0,True,0.030268,6.0,0.000000,0.000000,average_precision,...,1,tree,NaN,NaN,NaN,NaN,NaN,0.030268,0.030268,6
6,624,ecfp4_bit_624,0.029108,3.0,True,0.029108,7.0,0.000000,0.000000,average_precision,...,1,tree,NaN,NaN,NaN,NaN,NaN,0.029108,0.029108,7
7,262,ecfp4_bit_262,0.028130,4.0,True,0.028130,8.0,0.000140,0.001380,average_precision,...,1,tree,NaN,NaN,NaN,NaN,NaN,0.028130,0.028130,8
8,592,ecfp4_bit_592,0.027457,3.0,True,0.027457,9.0,0.011152,0.003453,average_precision,...,1,tree,NaN,NaN,NaN,NaN,NaN,0.027457,0.027457,9
9,1585,ecfp4_bit_1585,0.026178,12.0,True,0.026178,10.0,0.000394,0.000756,average_precision,...,1,tree,NaN,NaN,NaN,NaN,NaN,0.026178,0.026178,10


In [ ]:
# Fast reload of already-computed distribution-shift artifacts
# Use this cell when you do not want to rerun the heavy discriminator fitting cells.

# Heavy cells that can be skipped if the CSVs already exist:
#   - High-Capacity In-Sample Discrimination
#   - Same-Search Out-of-Sample Discrimination
#   - Extract List B: Fold-Discriminator Feature Importances

cached_paths = {
    "df_insample": OUT_ROOT / "cross_dataset_discrimination_high_capacity_insample.csv",
    "df_search_cv": OUT_ROOT / "cross_dataset_discrimination_same_search_cv.csv",
    "df_listB_main": OUT_ROOT
    / "cross_dataset_listB_same_search_cv_feature_importance.csv",
    "df_listB_insample": OUT_ROOT
    / "cross_dataset_listB_high_capacity_insample_feature_importance.csv",
}

missing_cached = [name for name, path in cached_paths.items() if not path.exists()]

if missing_cached:
    raise FileNotFoundError(
        "Missing cached artifacts. You must run the heavy discriminator cells first:\n"
        + "\n".join(f"- {name}: {cached_paths[name]}" for name in missing_cached)
    )

df_insample = pd.read_csv(cached_paths["df_insample"], low_memory=False).copy()
df_search_cv = pd.read_csv(cached_paths["df_search_cv"], low_memory=False).copy()
df_listB_main = pd.read_csv(cached_paths["df_listB_main"], low_memory=False).copy()
df_listB_insample = pd.read_csv(
    cached_paths["df_listB_insample"], low_memory=False
).copy()

# Main List B used downstream.
df_listB = df_listB_main.copy()

# Keep only the valid main datasets.
for name, df in {
    "df_insample": df_insample,
    "df_search_cv": df_search_cv,
    "df_listB_main": df_listB_main,
    "df_listB_insample": df_listB_insample,
    "df_listB": df_listB,
}.items():
    if "dataset" in df.columns:
        bad = set(df["dataset"]) - set(DATASETS_MAIN)
        if bad:
            print(f"{name}: dropping excluded datasets {sorted(bad)}")
            globals()[name] = df[df["dataset"].isin(DATASETS_MAIN)].copy()

print("Loaded cached shift/discriminator artifacts.")
print("df_insample      :", df_insample.shape)
print("df_search_cv     :", df_search_cv.shape)
print("df_listB_main    :", df_listB_main.shape)
print("df_listB_insample:", df_listB_insample.shape)
print("df_listB         :", df_listB.shape)

Loaded cached shift/discriminator artifacts.
df_insample      : (81, 16)
df_search_cv     : (63, 28)
df_listB_main    : (61758, 27)
df_listB_insample: (65664, 27)
df_listB         : (61758, 27)


## Fold-Aware List A vs List B Feature Overlap

Lo shift coinvolge le stesse feature del modello di attività?


In [ ]:
# Fold-Aware List A vs List B Feature Overlap
# Lo shift coinvolge le stesse feature del modello di attività?

PLOT_FPS = ["ecfp4", "maccs"]

TOP_K_VALUES_OVERLAP = [10, 20, 50, 100, 150, 200]

OVERLAP_K = 50
LISTB_SOURCE = "same_search_cv"

FP_N_FEATURES = {
    "ecfp4": 2048,
    "maccs": 167,
}

PAIR_TO_OUTER_FOLD = {
    "F1_vs_F2": 1,
    "F1_vs_F3": 2,
    "F2_vs_F3": 3,
}


def top_features_from_listB(
    df_b: pd.DataFrame,
    dataset: str,
    pair: str,
    model: str,
    fingerprint: str,
    source: str,
    k: int,
) -> set:
    sub = df_b[
        (df_b["dataset"] == dataset)
        & (df_b["pair"] == pair)
        & (df_b["model"] == model)
        & (df_b["fingerprint"] == fingerprint)
        & (df_b["source"] == source)
    ].copy()

    if sub.empty:
        return set()

    # Avoid artificial overlap from zero-importance features.
    sub["abs_importance"] = pd.to_numeric(sub["abs_importance"], errors="coerce")
    sub = sub[sub["abs_importance"] > 0].copy()

    if sub.empty:
        return set()

    sub = sub.sort_values(["rank", "feature_idx"], ascending=[True, True])
    return set(sub.head(k)["feature_idx"].astype(int))


def normalize_protocol_name(x: str) -> str:
    x = str(x).strip().lower()
    if x in {"ood holdout", "ood_holdout", "holdout_ood"}:
        return "ood"
    if x in {"random shuffle", "random_shuffle", "random"}:
        return "random"
    if "ood" in x and "random" not in x:
        return "ood"
    if "random" in x:
        return "random"
    return x


def top_features_listA_by_protocol_and_fold(
    df_a: pd.DataFrame,
    dataset: str,
    model: str,
    fingerprint: str,
    protocol: str,
    fold: int,
    k: int,
) -> set:
    work = df_a.copy()
    work["protocol_family"] = work["protocol"].map(normalize_protocol_name)
    target_protocol = normalize_protocol_name(protocol)

    sub = work[
        (work["dataset"] == dataset)
        & (work["model"] == model)
        & (work["fingerprint"] == fingerprint)
        & (work["protocol_family"] == target_protocol)
        & (work["fold"].astype(int) == int(fold))
    ].copy()

    if sub.empty:
        return set()

    # Avoid artificial overlap from zero-importance activity features.
    sub["activity_importance"] = pd.to_numeric(
        sub["activity_importance"],
        errors="coerce",
    )

    sub = sub[sub["activity_importance"] > 0].copy()

    if sub.empty:
        return set()

    agg = (
        sub.groupby("feature_idx", as_index=False)["activity_importance"]
        .mean()
        .sort_values(["activity_importance", "feature_idx"], ascending=[False, True])
    )

    return set(agg.head(k)["feature_idx"].astype(int))


PROTOCOLS = ["OOD holdout", "Random shuffle"]
PROTOCOL_SHORT = {"OOD holdout": "ood", "Random shuffle": "random"}

overlap_rows = []

for k in TOP_K_VALUES_OVERLAP:
    for dataset in DATASETS_MAIN:
        for fold_a, fold_b in PAIRS:
            pair_name = f"{fold_a}_vs_{fold_b}"
            outer_fold = PAIR_TO_OUTER_FOLD[pair_name]

            for fp in PLOT_FPS:
                n_features = FP_N_FEATURES[fp]

                for model in ["DT", "LR", "SVM"]:
                    b_features = top_features_from_listB(
                        df_b=df_listB,
                        dataset=dataset,
                        pair=pair_name,
                        model=model,
                        fingerprint=fp,
                        source=LISTB_SOURCE,
                        k=k,
                    )

                    if not b_features:
                        continue

                    for protocol in PROTOCOLS:
                        a_features = top_features_listA_by_protocol_and_fold(
                            df_a=df_listA,
                            dataset=dataset,
                            model=model,
                            fingerprint=fp,
                            protocol=protocol,
                            fold=outer_fold,
                            k=k,
                        )

                        if not a_features:
                            continue

                        common_features = a_features & b_features

                        n_activity_features = len(a_features)
                        n_shift_features = len(b_features)
                        n_overlap = len(common_features)

                        # Conservative denominator using the larger actually available list.
                        # This avoids treating a top-k list with many zero/tied features
                        # as if it contained k meaningful features.
                        effective_k = max(n_activity_features, n_shift_features)

                        if effective_k == 0:
                            continue

                        overlap = n_overlap / effective_k

                        expected_overlap_count_random = (
                            n_activity_features * n_shift_features / n_features
                            if n_features > 0
                            else np.nan
                        )

                        if (
                            n_features > 0
                            and n_activity_features > 0
                            and n_shift_features > 0
                        ):
                            # Probability of observing at least n_overlap shared features by chance.
                            hypergeom_p_value = hypergeom.sf(
                                n_overlap - 1,
                                n_features,
                                n_activity_features,
                                n_shift_features,
                            )
                        else:
                            hypergeom_p_value = np.nan

                        random_expected_overlap = (
                            expected_overlap_count_random / effective_k
                            if effective_k > 0
                            and np.isfinite(expected_overlap_count_random)
                            else np.nan
                        )

                        overlap_enrichment = (
                            overlap / random_expected_overlap
                            if random_expected_overlap > 0
                            else np.nan
                        )

                        overlap_rows.append(
                            {
                                "dataset": dataset,
                                "pair": pair_name,
                                "outer_fold": outer_fold,
                                "fingerprint": fp,
                                "model": model,
                                "activity_protocol": PROTOCOL_SHORT[protocol],
                                "k": k,
                                "effective_k": effective_k,
                                "n_features": n_features,
                                "n_activity_features": n_activity_features,
                                "n_shift_features": n_shift_features,
                                "overlap": overlap,
                                "n_overlap": n_overlap,
                                "random_expected_overlap": random_expected_overlap,
                                "expected_overlap_count_random": expected_overlap_count_random,
                                "overlap_enrichment": overlap_enrichment,
                                "hypergeom_p_value": hypergeom_p_value,
                                "overlap_features": json.dumps(
                                    sorted(list(common_features))
                                ),
                            }
                        )

df_overlap = pd.DataFrame(overlap_rows)

# Multi-k table
df_overlap.to_csv(
    OUT_ROOT / "cross_dataset_listA_listB_overlap_by_protocol_foldaware_multi_k.csv",
    index=False,
)

# Per-k tables for backward compatibility with downstream scripts/notebooks
for k in TOP_K_VALUES_OVERLAP:
    sub = df_overlap[df_overlap["k"] == k]
    sub.to_csv(
        OUT_ROOT
        / f"cross_dataset_listA_listB_overlap_by_protocol_foldaware_top{k}.csv",
        index=False,
    )

# Sanity: every k requested actually produced rows
present_k = sorted(df_overlap["k"].unique().tolist()) if len(df_overlap) else []
missing_k = [k for k in TOP_K_VALUES_OVERLAP if k not in present_k]
assert not missing_k, f"No overlap rows for k values: {missing_k}"

print(
    f"OK: df_overlap built for k in {TOP_K_VALUES_OVERLAP} ({len(df_overlap)} rows total)."
)

display(df_overlap.head(12))

OK: df_overlap built for k in [10, 20, 50, 100, 150, 200] (648 rows total).


,dataset,pair,outer_fold,fingerprint,model,activity_protocol,k,effective_k,n_features,n_activity_features,n_shift_features,overlap,n_overlap,random_expected_overlap,expected_overlap_count_random,overlap_enrichment,hypergeom_p_value,overlap_features
0,drd2,F1_vs_F2,1,ecfp4,DT,ood,10,10,2048,10,10,0.1,1,0.004883,0.048828,20.48,0.047872,[2000]
1,drd2,F1_vs_F2,1,ecfp4,DT,random,10,10,2048,10,10,0.1,1,0.004883,0.048828,20.48,0.047872,[2000]
2,drd2,F1_vs_F2,1,ecfp4,LR,ood,10,10,2048,10,10,0.0,0,0.004883,0.048828,0.00,1.000000,[]
3,drd2,F1_vs_F2,1,ecfp4,LR,random,10,10,2048,10,10,0.0,0,0.004883,0.048828,0.00,1.000000,[]
4,drd2,F1_vs_F2,1,ecfp4,SVM,ood,10,10,2048,10,10,0.0,0,0.004883,0.048828,0.00,1.000000,[]
5,drd2,F1_vs_F2,1,ecfp4,SVM,random,10,10,2048,10,10,0.0,0,0.004883,0.048828,0.00,1.000000,[]
6,drd2,F1_vs_F2,1,maccs,DT,ood,10,10,167,10,10,0.2,2,0.059880,0.598802,3.34,0.112307,"[82, 90]"
7,drd2,F1_vs_F2,1,maccs,DT,random,10,10,167,10,10,0.2,2,0.059880,0.598802,3.34,0.112307,"[77, 104]"
8,drd2,F1_vs_F2,1,maccs,LR,ood,10,10,167,10,10,0.1,1,0.059880,0.598802,1.67,0.470248,[85]
9,drd2,F1_vs_F2,1,maccs,LR,random,10,10,167,10,10,0.1,1,0.059880,0.598802,1.67,0.470248,[85]


## Load OOD-vs-Random Per-Fold Protocol Results


In [ ]:
OOD_CROSS_DIR = (
    PROJECT_ROOT / "results" / "results_ood_vs_random_shuffle" / TASK / "cross_dataset"
)

PROTOCOL_PATH = OOD_CROSS_DIR / "cross_dataset_protocol_per_fold.csv"

if not PROTOCOL_PATH.exists():
    raise FileNotFoundError(
        f"Missing protocol table from OOD-vs-random study: {PROTOCOL_PATH}"
    )

df_protocol_per_fold = pd.read_csv(PROTOCOL_PATH).copy()
df_protocol_per_fold = df_protocol_per_fold[
    df_protocol_per_fold["dataset"].isin(DATASETS_MAIN)
].copy()

assert "kdr" not in set(
    df_protocol_per_fold["dataset"]
), "KDR should not enter the main shift-vs-protocol analysis."

print("protocol_per_fold columns:")
print(df_protocol_per_fold.columns.tolist())

display(df_protocol_per_fold.head())

protocol_per_fold columns:
['dataset', 'dataset_label', 'model', 'model_short', 'fingerprint', 'protocol', 'fold', 'inner_pr_auc', 'inner_train_pr_auc', 'train_pr_auc', 'test_pr_auc', 'inner_test_gap', 'train_test_gap', 'dataset_order', 'model_order', 'fingerprint_order', 'protocol_order']


,dataset,dataset_label,model,model_short,fingerprint,protocol,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap,dataset_order,model_order,fingerprint_order,protocol_order
0,drd2,DRD2,Decision Tree,DT,ECFP4,OOD holdout,1,0.849198,0.931822,0.9496,0.6620,0.187198,0.2876,0,0,0,0
1,drd2,DRD2,Decision Tree,DT,ECFP4,OOD holdout,2,0.697409,0.882575,0.7683,0.8022,-0.104791,-0.0339,0,0,0,0
2,drd2,DRD2,Decision Tree,DT,ECFP4,OOD holdout,3,0.731996,0.945777,0.9029,0.7052,0.026796,0.1977,0,0,0,0
3,drd2,DRD2,Decision Tree,DT,ECFP4,Random shuffle,1,0.906735,0.955275,0.9339,0.6687,0.238035,0.2652,0,0,0,1
4,drd2,DRD2,Decision Tree,DT,ECFP4,Random shuffle,2,0.830900,0.931783,0.8539,0.7398,0.091100,0.1141,0,0,0,1


## Pairwise Shift vs OOD-vs-Random Protocol Gap


In [ ]:
def infer_fold_column(df: pd.DataFrame) -> str:
    candidates = ["fold", "outer_fold", "fold_id", "test_fold"]

    for col in candidates:
        if col in df.columns:
            return col

    raise ValueError(
        "Could not infer fold column. " f"Available columns: {df.columns.tolist()}"
    )

## Pairwise shift vs OOD-vs-random validation gap


In [ ]:
# Pairwise shift vs OOD-vs-random validation gap
#
# Two complementary gaps, both built from protocol_per_fold.csv:
#
# 1. optimism_gap_diff:
#       optimism_gap_random  =  inner_pr_auc_random  - test_pr_auc_random
#       optimism_gap_ood     =  inner_pr_auc_ood     - test_pr_auc_ood
#       optimism_gap_diff    =  optimism_gap_random  - optimism_gap_ood
#    Positive value: random validation is more optimistic than OOD validation.
#
# 2. test_gap:
#       test_gap = test_pr_auc_ood - test_pr_auc_random
#    Positive value: OOD-selected model beats random-selected model on the real OOD test set.
# ============================================================

PLOT_FPS = ["ecfp4", "maccs"]

OUTER_FOLD_TO_PAIR = {
    1: "F1_vs_F2",
    2: "F1_vs_F3",
    3: "F2_vs_F3",
}

# Shift table.
# Main out-of-sample metric requested by the professor:
#   CV balanced accuracy.
#
# We keep shift_score_01 and proxy_A_distance as derived quantities.
df_shift_pair = (
    df_search_cv.query("fingerprint in @PLOT_FPS")
    .loc[lambda d: d["dataset"].isin(DATASETS_MAIN)]
    .groupby(["dataset", "pair"], as_index=False)
    .agg(
        balanced_accuracy_mean=("cv_balanced_accuracy_mean", "mean"),
        balanced_accuracy_max=("cv_balanced_accuracy_mean", "max"),
        shift_score_mean=("shift_score_01", "mean"),
        shift_score_max=("shift_score_01", "max"),
        proxy_A_distance_mean=("proxy_A_distance", "mean"),
        proxy_A_distance_max=("proxy_A_distance", "max"),
    )
)

# Main x-axis score for professor-facing plots.
df_shift_pair["balanced_accuracy"] = df_shift_pair["balanced_accuracy_max"]

# Derived score kept for backward compatibility with previous figures.
df_shift_pair["shift_score_01"] = df_shift_pair["shift_score_max"]

# Build the per-fold protocol table
df_gap_raw = df_protocol_per_fold.copy()
df_gap_raw["fingerprint_norm"] = df_gap_raw["fingerprint"].astype(str).str.lower()
df_gap_raw["model_short_norm"] = df_gap_raw["model_short"].astype(str).str.upper()

df_gap_raw = df_gap_raw[
    df_gap_raw["fingerprint_norm"].isin(PLOT_FPS)
    & df_gap_raw["model_short_norm"].isin(["DT", "LR", "SVM"])
].copy()

df_gap_raw["protocol_norm"] = df_gap_raw["protocol"].astype(str).str.lower()


def protocol_family(x: str) -> str:
    if "random" in x:
        return "random"
    if "ood" in x or "holdout" in x:
        return "ood"
    return x


df_gap_raw["protocol_family"] = df_gap_raw["protocol_norm"].map(protocol_family)
df_gap_raw = df_gap_raw[df_gap_raw["protocol_family"].isin(["random", "ood"])].copy()

df_gap_raw["outer_fold"] = df_gap_raw["fold"].astype(int)
df_gap_raw["pair"] = df_gap_raw["outer_fold"].map(OUTER_FOLD_TO_PAIR)

# Per-fold optimism gap, per protocol
df_gap_raw["optimism_gap"] = df_gap_raw["inner_pr_auc"] - df_gap_raw["test_pr_auc"]

df_gap_agg = df_gap_raw.groupby(
    ["dataset", "pair", "protocol_family"], as_index=False
).agg(
    inner_pr_auc=("inner_pr_auc", "mean"),
    test_pr_auc=("test_pr_auc", "mean"),
    optimism_gap=("optimism_gap", "mean"),
)

# Pivot random vs OOD
df_gap_pivot = df_gap_agg.pivot_table(
    index=["dataset", "pair"],
    columns="protocol_family",
    values=["inner_pr_auc", "test_pr_auc", "optimism_gap"],
).reset_index()

df_gap_pivot.columns = [
    "_".join(c).rstrip("_") if isinstance(c, tuple) else c for c in df_gap_pivot.columns
]

df_gap_pivot["optimism_gap_diff"] = (
    df_gap_pivot["optimism_gap_random"] - df_gap_pivot["optimism_gap_ood"]
)

df_gap_pivot["test_gap"] = (
    df_gap_pivot["test_pr_auc_ood"] - df_gap_pivot["test_pr_auc_random"]
)

df_gap_pair = df_gap_pivot[
    [
        "dataset",
        "pair",
        "inner_pr_auc_random",
        "inner_pr_auc_ood",
        "test_pr_auc_random",
        "test_pr_auc_ood",
        "optimism_gap_random",
        "optimism_gap_ood",
        "optimism_gap_diff",
        "test_gap",
    ]
].copy()

df_shift_gap = df_shift_pair.merge(df_gap_pair, on=["dataset", "pair"], how="inner")

df_shift_gap = df_shift_gap[df_shift_gap["dataset"].isin(DATASETS_MAIN)].copy()

assert "kdr" not in set(
    df_shift_gap["dataset"]
), "KDR should not enter the main shift-vs-protocol analysis."

expected_pairs = pd.MultiIndex.from_product(
    [DATASETS_MAIN, [f"{a}_vs_{b}" for a, b in PAIRS]],
    names=["dataset", "pair"],
).to_frame(index=False)

observed_pairs = df_shift_gap[["dataset", "pair"]].drop_duplicates()

missing_pairs = (
    expected_pairs.merge(
        observed_pairs, on=["dataset", "pair"], how="left", indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
)

if len(missing_pairs) > 0:
    raise ValueError(
        "Missing dataset/pair rows in df_shift_gap:\n"
        + missing_pairs.to_string(index=False)
    )

if len(df_shift_gap) != len(expected_pairs):
    print(
        f"WARNING: expected {len(expected_pairs)} dataset/pair rows, "
        f"got {len(df_shift_gap)}. There may be duplicated rows."
    )

df_shift_gap.to_csv(
    OUT_ROOT / "cross_dataset_pairwise_shift_vs_protocol_gap.csv",
    index=False,
)

display(df_shift_gap.round(3))

,dataset,pair,balanced_accuracy_mean,balanced_accuracy_max,shift_score_mean,shift_score_max,proxy_A_distance_mean,proxy_A_distance_max,balanced_accuracy,shift_score_01,inner_pr_auc_random,inner_pr_auc_ood,test_pr_auc_random,test_pr_auc_ood,optimism_gap_random,optimism_gap_ood,optimism_gap_diff,test_gap
0,drd2,F1_vs_F2,0.957,0.988,0.913,0.977,1.827,1.953,0.988,0.977,0.904,0.834,0.648,0.652,0.257,0.182,0.075,0.005
1,drd2,F1_vs_F3,0.939,0.971,0.879,0.941,1.757,1.882,0.971,0.941,0.851,0.698,0.804,0.806,0.047,-0.108,0.155,0.002
2,drd2,F2_vs_F3,0.963,0.986,0.926,0.971,1.852,1.943,0.986,0.971,0.879,0.719,0.750,0.735,0.128,-0.016,0.144,-0.016
3,hiv,F1_vs_F2,0.718,0.777,0.437,0.554,0.873,1.108,0.777,0.554,0.298,0.114,0.127,0.112,0.171,0.002,0.169,-0.015
4,hiv,F1_vs_F3,0.702,0.746,0.404,0.492,0.809,0.984,0.746,0.492,0.297,0.175,0.074,0.081,0.224,0.094,0.130,0.008
5,hiv,F2_vs_F3,0.753,0.805,0.506,0.610,1.012,1.220,0.805,0.610,0.394,0.162,0.033,0.033,0.361,0.129,0.232,-0.000
6,sol,F1_vs_F2,0.602,0.659,0.203,0.318,0.407,0.635,0.659,0.318,0.502,0.446,0.452,0.455,0.050,-0.009,0.058,0.002
7,sol,F1_vs_F3,0.667,0.718,0.334,0.436,0.668,0.871,0.718,0.436,0.520,0.468,0.451,0.445,0.070,0.023,0.046,-0.006
8,sol,F2_vs_F3,0.659,0.709,0.317,0.419,0.635,0.838,0.709,0.419,0.524,0.461,0.416,0.428,0.108,0.033,0.075,0.012


## Predictive-shift overlap vs OOD validation usefulness

Question:
Does OOD validation help more when the fold shift affects the same
features used by the activity model?


In [ ]:
# Predictive-shift overlap vs OOD validation usefulness
# Question: Does OOD validation help more when the fold shift affects the same features used by the activity model?

FIG_ROOT = OUT_ROOT / "figures"
FIG_ROOT.mkdir(parents=True, exist_ok=True)

PLOT_FPS = ["ecfp4", "maccs"]
PLOT_MODELS = ["DT", "LR", "SVM"]

OVERLAP_K = 20

FP_N_FEATURES = {
    "ecfp4": 2048,
    "maccs": 167,
}

OUTER_FOLD_TO_PAIR = {
    1: "F1_vs_F2",
    2: "F1_vs_F3",
    3: "F2_vs_F3",
}

PAIR_LABELS = {
    "F1_vs_F2": "F1–F2",
    "F1_vs_F3": "F1–F3",
    "F2_vs_F3": "F2–F3",
}

DATASET_COLORS = {
    "drd2": "#4C78A8",
    "hiv": "#F58518",
    "sol": "#B279A2",
}

PAIR_MARKERS = {
    "F1_vs_F2": "o",
    "F1_vs_F3": "s",
    "F2_vs_F3": "^",
}


def _protocol_family(x):
    x = str(x).strip().lower()

    if "random" in x:
        return "random"

    if "ood" in x or "holdout" in x:
        return "ood"

    return x


# Build List A / List B overlap enrichment table

ov = df_overlap.copy()

ov = ov[
    ov["dataset"].isin(DATASETS_MAIN)
    & ov["fingerprint"].isin(PLOT_FPS)
    & ov["model"].isin(PLOT_MODELS)
    & (ov["k"].astype(int) == int(OVERLAP_K))
].copy()

ov["fingerprint_norm"] = ov["fingerprint"].astype(str).str.lower()
ov["model_norm"] = ov["model"].astype(str).str.upper()
ov["activity_protocol_norm"] = ov["activity_protocol"].astype(str).str.lower()

# Use the already computed overlap and enrichment from the overlap table.
# This preserves the corrected denominator and random expectation.
ov_pivot = ov.pivot_table(
    index=["dataset", "pair", "fingerprint_norm", "model_norm"],
    columns="activity_protocol_norm",
    values=["overlap", "overlap_enrichment"],
    aggfunc="mean",
).reset_index()

ov_pivot.columns = [
    "_".join(c).rstrip("_") if isinstance(c, tuple) else c for c in ov_pivot.columns
]

# Ensure expected protocol columns exist.
for col in [
    "overlap_ood",
    "overlap_random",
    "overlap_enrichment_ood",
    "overlap_enrichment_random",
]:
    if col not in ov_pivot.columns:
        ov_pivot[col] = np.nan

ov_pivot = ov_pivot.rename(
    columns={
        "overlap_enrichment_ood": "overlap_ood_enrichment",
        "overlap_enrichment_random": "overlap_random_enrichment",
    }
)

ov_pivot["overlap_mean"] = ov_pivot[["overlap_ood", "overlap_random"]].mean(
    axis=1,
    skipna=True,
)

ov_pivot["overlap_mean_enrichment"] = ov_pivot[
    ["overlap_ood_enrichment", "overlap_random_enrichment"]
].mean(axis=1, skipna=True)

ov_pivot["n_overlap_protocols"] = (
    ov_pivot[["overlap_ood", "overlap_random"]].notna().sum(axis=1)
)

if (ov_pivot["n_overlap_protocols"] < 2).any():
    print(
        "WARNING: some model/fingerprint rows have overlap for only one activity protocol."
    )
    display(
        ov_pivot.loc[
            ov_pivot["n_overlap_protocols"] < 2,
            [
                "dataset",
                "pair",
                "fingerprint_norm",
                "model_norm",
                "overlap_ood",
                "overlap_random",
            ],
        ].head(20)
    )


# Build model/fingerprint-level OOD test benefit table

gap = df_protocol_per_fold.copy()

gap = gap[gap["dataset"].isin(DATASETS_MAIN)].copy()

gap["fingerprint_norm"] = gap["fingerprint"].astype(str).str.lower()
gap["model_norm"] = gap["model_short"].astype(str).str.upper()

gap = gap[
    gap["fingerprint_norm"].isin(PLOT_FPS) & gap["model_norm"].isin(PLOT_MODELS)
].copy()

gap["protocol_family"] = gap["protocol"].map(_protocol_family)
gap = gap[gap["protocol_family"].isin(["ood", "random"])].copy()

gap["outer_fold"] = gap["fold"].astype(int)
gap["pair"] = gap["outer_fold"].map(OUTER_FOLD_TO_PAIR)

if gap["pair"].isna().any():
    bad_folds = sorted(gap.loc[gap["pair"].isna(), "outer_fold"].unique())
    raise ValueError(f"Could not map these outer folds to fold pairs: {bad_folds}")

gap["optimism_gap"] = gap["inner_pr_auc"] - gap["test_pr_auc"]

gap_agg = gap.groupby(
    ["dataset", "pair", "fingerprint_norm", "model_norm", "protocol_family"],
    as_index=False,
).agg(
    inner_pr_auc=("inner_pr_auc", "mean"),
    test_pr_auc=("test_pr_auc", "mean"),
    optimism_gap=("optimism_gap", "mean"),
)

gap_pivot = gap_agg.pivot_table(
    index=["dataset", "pair", "fingerprint_norm", "model_norm"],
    columns="protocol_family",
    values=["inner_pr_auc", "test_pr_auc", "optimism_gap"],
    aggfunc="mean",
).reset_index()

gap_pivot.columns = [
    "_".join(c).rstrip("_") if isinstance(c, tuple) else c for c in gap_pivot.columns
]

required_gap_cols = [
    "test_pr_auc_ood",
    "test_pr_auc_random",
    "optimism_gap_ood",
    "optimism_gap_random",
]

missing_gap_cols = [c for c in required_gap_cols if c not in gap_pivot.columns]

if missing_gap_cols:
    raise ValueError(f"Missing required protocol columns: {missing_gap_cols}")

gap_pivot["test_benefit"] = (
    gap_pivot["test_pr_auc_ood"] - gap_pivot["test_pr_auc_random"]
)

gap_pivot["optimism_reduction"] = (
    gap_pivot["optimism_gap_random"] - gap_pivot["optimism_gap_ood"]
)


# Add structural shift score for each model/fingerprint row

shift_mfp = df_search_cv.copy()

shift_mfp = shift_mfp[
    shift_mfp["dataset"].isin(DATASETS_MAIN)
    & shift_mfp["fingerprint"].isin(PLOT_FPS)
    & shift_mfp["model"].isin(PLOT_MODELS)
].copy()

shift_mfp["fingerprint_norm"] = shift_mfp["fingerprint"].astype(str).str.lower()
shift_mfp["model_norm"] = shift_mfp["model"].astype(str).str.upper()

shift_mfp = shift_mfp.groupby(
    ["dataset", "pair", "fingerprint_norm", "model_norm"],
    as_index=False,
).agg(
    balanced_accuracy=("cv_balanced_accuracy_mean", "mean"),
    shift_score=("shift_score_01", "mean"),
    proxy_A_distance=("proxy_A_distance", "mean"),
)


# Merge into one granular table

predictive_shift_granular = ov_pivot.merge(
    gap_pivot,
    on=["dataset", "pair", "fingerprint_norm", "model_norm"],
    how="inner",
).merge(
    shift_mfp,
    on=["dataset", "pair", "fingerprint_norm", "model_norm"],
    how="left",
)

if len(predictive_shift_granular) == 0:
    raise ValueError(
        "No rows after merging overlap, protocol gap, and shift score tables."
    )

predictive_shift_granular.to_csv(
    OUT_ROOT / f"predictive_shift_overlap_granular_top{OVERLAP_K}.csv",
    index=False,
)


# Aggregate to dataset × fold-pair for a clean main plot

plot_df = predictive_shift_granular.groupby(["dataset", "pair"], as_index=False).agg(
    overlap_ood_mean=("overlap_ood", "mean"),
    overlap_ood_enrichment_mean=("overlap_ood_enrichment", "mean"),
    overlap_mean=("overlap_mean", "mean"),
    overlap_mean_enrichment_mean=("overlap_mean_enrichment", "mean"),
    test_benefit_mean=("test_benefit", "mean"),
    optimism_reduction_mean=("optimism_reduction", "mean"),
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
    balanced_accuracy_max=("balanced_accuracy", "max"),
    shift_score_mean=("shift_score", "mean"),
    shift_score_max=("shift_score", "max"),
    n_rows=("test_benefit", "size"),
)

plot_df["dataset_label"] = (
    plot_df["dataset"].map(DATASET_LABELS).fillna(plot_df["dataset"])
)

plot_df["pair_label"] = plot_df["pair"].map(PAIR_LABELS).fillna(plot_df["pair"])

plot_df.to_csv(
    OUT_ROOT / f"predictive_shift_overlap_pair_summary_top{OVERLAP_K}.csv",
    index=False,
)

display(predictive_shift_granular.head())
display(plot_df.round(3))

,dataset,pair,fingerprint_norm,model_norm,overlap_ood,overlap_random,overlap_ood_enrichment,overlap_random_enrichment,overlap_mean,overlap_mean_enrichment,...,inner_pr_auc_random,optimism_gap_ood,optimism_gap_random,test_pr_auc_ood,test_pr_auc_random,test_benefit,optimism_reduction,balanced_accuracy,shift_score,proxy_A_distance
0,drd2,F1_vs_F2,ecfp4,DT,0.10,0.05,10.2400,5.12,0.075,7.68000,...,0.906735,0.187198,0.238035,0.6620,0.6687,-0.0067,0.050838,0.965622,0.931244,1.862487
1,drd2,F1_vs_F2,ecfp4,LR,0.00,0.00,0.0000,0.00,0.000,0.00000,...,0.942367,0.178507,0.277467,0.6842,0.6649,0.0193,0.098960,0.988265,0.976530,1.953061
2,drd2,F1_vs_F2,ecfp4,SVM,0.05,0.00,5.1200,0.00,0.025,2.56000,...,0.935004,0.182189,0.276304,0.6519,0.6587,-0.0068,0.094115,0.988269,0.976537,1.953075
3,drd2,F1_vs_F2,maccs,DT,0.25,0.20,2.0875,1.67,0.225,1.87875,...,0.891284,0.216497,0.287884,0.6262,0.6034,0.0228,0.071387,0.942553,0.885106,1.770212
4,drd2,F1_vs_F2,maccs,LR,0.20,0.20,1.6700,1.67,0.200,1.67000,...,0.877967,0.163956,0.244067,0.6403,0.6339,0.0064,0.080110,0.926618,0.853237,1.706473


,dataset,pair,overlap_ood_mean,overlap_ood_enrichment_mean,overlap_mean,overlap_mean_enrichment_mean,test_benefit_mean,optimism_reduction_mean,balanced_accuracy_mean,balanced_accuracy_max,shift_score_mean,shift_score_max,n_rows,dataset_label,pair_label
0,drd2,F1_vs_F2,0.158,3.673,0.142,2.750,0.005,0.075,0.957,0.988,0.913,0.977,6,DRD2,F1–F2
1,drd2,F1_vs_F3,0.158,2.106,0.158,2.106,0.002,0.155,0.939,0.971,0.879,0.941,6,DRD2,F1–F3
2,drd2,F2_vs_F3,0.117,5.102,0.108,2.968,-0.016,0.144,0.963,0.986,0.926,0.971,6,DRD2,F2–F3
3,hiv,F1_vs_F2,0.133,2.681,0.117,2.542,-0.015,0.169,0.718,0.777,0.437,0.554,6,HIV,F1–F2
4,hiv,F1_vs_F3,0.142,2.180,0.146,2.108,0.008,0.130,0.702,0.746,0.404,0.492,6,HIV,F1–F3
5,hiv,F2_vs_F3,0.100,0.835,0.087,1.122,-0.000,0.232,0.753,0.805,0.506,0.610,6,HIV,F2–F3
6,sol,F1_vs_F2,0.058,2.091,0.050,1.612,0.002,0.058,0.602,0.659,0.203,0.318,6,Sol,F1–F2
7,sol,F1_vs_F3,0.100,0.835,0.079,0.661,-0.006,0.046,0.667,0.718,0.334,0.436,6,Sol,F1–F3
8,sol,F2_vs_F3,0.050,0.418,0.067,0.949,0.012,0.075,0.659,0.709,0.317,0.419,6,Sol,F2–F3
